# 30 Building an ETL Pipeline

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Combine everything from lessons 01–29 into a small but production-shaped pipeline: ingest raw CSV and JSON files into <b>Bronze</b>, clean, deduplicate, validate and enrich them into <b>Silver</b> (with a quarantine table), aggregate into <b>Gold</b>, reconcile the counts, and prove the pipeline is <b>idempotent</b> by running it twice.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Interviewers rarely ask about one function in isolation. They ask "walk me through a pipeline you built" and "what happens if it runs twice?". This lesson is a compact version of the medallion pipelines you'll build later with Delta, Auto Loader and Lakeflow, and it's a template you can reuse for your own projects.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 The medallion architecture</b><br>
<code>`</code><br> raw files (volume)          BRONZE                     SILVER                       GOLD<br> ─────────────────          ─────────────              ──────────────               ──────────────<br> orders_*.csv     ──read──▶ bronze_orders   ──clean──▶ silver_orders   ──agg────▶ gold_daily_sales<br> customers.json   ──read──▶ bronze_customers  dedupe   (typed, valid,              gold_customer_value<br>                            (as received +   validate  deduplicated,<br>                             audit columns)  enrich    enriched)<br>                                                 │<br>                                                 └────▶ quarantine_orders (bad rows + reasons)<br><code>`</code>
<ul><li><b>Bronze</b>: raw data as received, plus audit columns (source file, ingestion time). Nothing is thrown away</li><li><b>Silver</b>: typed, cleaned, deduplicated, validated, conformed. One row per business entity</li><li><b>Gold</b>: business-level aggregates and marts, ready for dashboards</li><li><b>Idempotent</b>: running the same batch again gives the same result, with no duplicates</li></ul>
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A small online shop receives one orders CSV per day from its checkout system and a customers JSON export from its CRM. The finance dashboard needs daily revenue per country and the lifetime value of each customer. Sometimes a file is re-sent, a row is duplicated, an amount is <code>N/A</code>, or an order references a customer the CRM doesn't know. This pipeline handles all of that, and can be re-run safely after a failure.
</div>

## Pipeline design

| Step | Input | Output | Key techniques (lesson) |
|---|---|---|---|
| Extract | CSV and JSON files in a volume | Bronze tables | Explicit schemas (04), CSV options and corrupt records (21), JSON (22), `_metadata` lineage (21) |
| Clean | Bronze orders | Typed orders | `try_cast` (04, 28), `trim`/`upper` (14), dates (15), nulls (16) |
| Deduplicate | Typed orders | One row per order | `row_number` latest record (10, 19) |
| Validate | Deduplicated orders | Silver + quarantine | Rules with reasons (29) |
| Enrich | Silver + customers | Silver with country and tier | Left join (17) |
| Aggregate | Silver | Gold tables | `groupBy`/`agg` (11, 12), windows (19) |
| Load | DataFrames | Delta tables | Dynamic partition overwrite for idempotency (24, 25) |
| Reconcile | All layers | Pass/fail report | Counts must add up (29) |

## Setup: schema, volume and configuration

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates the schema and volume, and defines the configuration in one dictionary: paths, table names and quality thresholds.<br><br>
<b>Why it matters</b><br>Keeping configuration separate from logic is what lets the same code run in dev and prod (different catalog), or for a different date. In a job, these values would come from job parameters or widgets.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The configuration printed as a dictionary.
</div>

In [0]:
CATALOG, SCHEMA, VOLUME = "workspace", "spark_basics", "raw_files"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
BASE = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/lesson30"
DB = f"{CATALOG}.{SCHEMA}"

In [0]:
import logging, sys
from pyspark.sql import functions as F, Window

CONFIG = {
    "landing_orders": f"{BASE}/landing/orders",
    "landing_customers": f"{BASE}/landing/customers",
    "bronze_orders": f"{DB}.lesson30_bronze_orders",
    "bronze_customers": f"{DB}.lesson30_bronze_customers",
    "silver_orders": f"{DB}.lesson30_silver_orders",
    "quarantine_orders": f"{DB}.lesson30_quarantine_orders",
    "gold_daily_sales": f"{DB}.lesson30_gold_daily_sales",
    "gold_customer_value": f"{DB}.lesson30_gold_customer_value",
    "max_bad_ratio": 0.5,
}

logger = logging.getLogger("lesson30")
if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s"))
    logger.addHandler(handler)
logger.setLevel(logging.INFO)
logger.propagate = False

for k, v in CONFIG.items():
    print(f"{k:<20} {v}")

## Land the raw files

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes two daily orders CSV files and one customers JSON file to the landing folder, as a source system would.<br><br>
<b>Why it matters</b><br>The data deliberately contains every problem the pipeline must handle: an order sent twice with an update, a bad amount, a bad date, lowercase country codes, an unknown customer, and a malformed line.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Three files listed under <code>landing/</code>.
</div>

In [0]:
import os, json

def write_file(path, text):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)

write_file(f"{CONFIG['landing_orders']}/orders_2024-11-01.csv", """order_id,customer_id,order_ts,amount,currency,status
1001,C1,2024-11-01 09:15:00,120.50,inr,PAID
1002,C2,2024-11-01 10:00:00,80.00,GBP,PAID
1003,C3,2024-11-01 11:30:00,N/A,AED,PAID
1004,C1,2024-11-01 12:45:00,35.00,INR,CANCELLED
1005,C9,2024-11-01 13:00:00,60.00,USD,PAID
""")

write_file(f"{CONFIG['landing_orders']}/orders_2024-11-02.csv", """order_id,customer_id,order_ts,amount,currency,status
1002,C2,2024-11-01 10:00:00,85.00,GBP,PAID
1006,C4,2024-11-02 08:20:00,300.00,AED,PAID
1007,C2,2024-11-02 09:10:00,15.00,GBP,REFUND
1008,C3,02/11/2024 10:00,45.00,AED,PAID
this line is broken
1009,C1,2024-11-02 18:30:00,210.00,INR,PAID
""")

customers_raw = [
    {"customer_id": "C1", "name": "Amal", "country": "IN", "tier": "gold", "updated_at": "2024-10-01T00:00:00"},
    {"customer_id": "C2", "name": "John", "country": "UK", "tier": "silver", "updated_at": "2024-10-05T00:00:00"},
    {"customer_id": "C3", "name": "Vivek", "country": "AE", "tier": "gold", "updated_at": "2024-10-07T00:00:00"},
    {"customer_id": "C4", "name": "Sara", "country": "AE", "tier": "bronze", "updated_at": "2024-10-09T00:00:00"},
]
write_file(f"{CONFIG['landing_customers']}/customers.json", "\n".join(json.dumps(c) for c in customers_raw) + "\n")

for root, _, files in os.walk(f"{BASE}/landing"):
    for f in sorted(files):
        if not f.startswith("."):
            print(os.path.join(root, f).replace(BASE, "BASE"))

## Step 1: Extract into Bronze

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Reads the raw files with explicit schemas (everything as <code>STRING</code> for the CSV, so nothing is lost), keeps malformed lines in <code>_corrupt_record</code>, adds audit columns, and writes the Bronze tables.<br><br>
<b>Why it matters</b><br>Bronze is your <b>replayable</b> copy of what arrived. Reading everything as strings means no value is rejected at this stage. Typing happens in Silver, where problems can be quarantined. Partitioning Bronze by <code>batch_date</code> and overwriting only the dates in this batch makes re-runs safe.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Bronze orders: 11 rows (5 + 6, including the broken line), with <code>source_file</code>, <code>batch_date</code> and <code>ingested_at</code>. Bronze customers: 4 rows.
</div>

In [0]:
BRONZE_ORDERS_SCHEMA = """order_id STRING, customer_id STRING, order_ts STRING, amount STRING,
                          currency STRING, status STRING, _corrupt_record STRING"""

def extract_orders(path):
    return (
        spark.read.format("csv")
        .option("header", True)
        .option("mode", "PERMISSIVE")
        .option("columnNameOfCorruptRecord", "_corrupt_record")
        .schema(BRONZE_ORDERS_SCHEMA)
        .load(path)
        .withColumns({
            "source_file": F.col("_metadata.file_name"),
            "batch_date": F.to_date(F.regexp_extract(F.col("_metadata.file_name"), r"(\d{4}-\d{2}-\d{2})", 1)),
            "ingested_at": F.current_timestamp(),
        })
    )

def extract_customers(path):
    return (
        spark.read.schema("customer_id STRING, name STRING, country STRING, tier STRING, updated_at STRING")
        .json(path)
        .withColumns({"source_file": F.col("_metadata.file_name"), "ingested_at": F.current_timestamp()})
    )

def load_bronze():
    orders = extract_orders(CONFIG["landing_orders"])
    (orders.write.mode("overwrite").option("partitionOverwriteMode", "dynamic")
           .partitionBy("batch_date").saveAsTable(CONFIG["bronze_orders"]))
    extract_customers(CONFIG["landing_customers"]).write.mode("overwrite").saveAsTable(CONFIG["bronze_customers"])
    logger.info("bronze: %d orders, %d customers",
                spark.table(CONFIG["bronze_orders"]).count(), spark.table(CONFIG["bronze_customers"]).count())

load_bronze()
spark.table(CONFIG["bronze_orders"]).orderBy("source_file", "order_id").show(truncate=False)

## Step 2: Transform into Silver

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Builds the Silver orders in clear stages: type and standardize, deduplicate (latest version of each order wins), validate with reasons, enrich with customer attributes, then split good rows from quarantined ones.<br><br>
<b>Why it matters</b><br>Each stage is a small, named, testable function, which is how production code should look. The bad rows aren't dropped: they go to quarantine <b>with the reasons</b>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
No output yet: the functions are defined.
</div>

In [0]:
STATUSES = ["PAID", "CANCELLED", "REFUND"]

def standardize(bronze):
    """Types and cleans every column. try_* functions turn bad values into nulls instead of failing."""
    return (
        bronze
        .withColumns({
            "order_id": F.col("order_id").try_cast("int"),
            "customer_id": F.upper(F.trim("customer_id")),
            "order_ts": F.try_to_timestamp("order_ts", F.lit("yyyy-MM-dd HH:mm:ss")),
            "amount": F.col("amount").try_cast("decimal(10,2)"),
            "currency": F.upper(F.trim("currency")),
            "status": F.upper(F.trim("status")),
        })
        .withColumn("order_date", F.to_date("order_ts"))
    )

def deduplicate(df):
    """Keeps the most recently ingested version of each order (the later file wins)."""
    w = Window.partitionBy("order_id").orderBy(F.col("batch_date").desc(), F.col("source_file").desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

def validate(df):
    """Adds a dq_reasons array: empty means the row is valid."""
    rules = {
        "malformed line": F.col("_corrupt_record").isNotNull() & F.col("order_id").isNull(),
        "invalid amount": F.col("amount").isNull(),
        "invalid timestamp": F.col("order_ts").isNull(),
        "unknown status": ~F.col("status").isin(STATUSES),
        "unknown customer": F.col("customer_name").isNull(),
    }
    return df.withColumn("dq_reasons", F.array_compact(F.array(*[F.when(c, F.lit(n)) for n, c in rules.items()])))

def latest_customers():
    w = Window.partitionBy("customer_id").orderBy(F.col("updated_at").desc())
    return (
        spark.table(CONFIG["bronze_customers"])
        .withColumn("_rn", F.row_number().over(w)).filter("_rn = 1")
        .select("customer_id", F.col("name").alias("customer_name"), "country", "tier")
    )

def build_silver():
    bronze = spark.table(CONFIG["bronze_orders"])
    enriched = deduplicate(standardize(bronze)).join(latest_customers(), "customer_id", "left")
    checked = validate(enriched)
    good = checked.filter(F.size("dq_reasons") == 0).select(
        "order_id", "customer_id", "customer_name", "country", "tier",
        "order_ts", "order_date", "amount", "currency", "status", "source_file", "batch_date",
    )
    bad = checked.filter(F.size("dq_reasons") > 0).select(
        "order_id", "customer_id", "amount", "order_ts", "status", "dq_reasons", "_corrupt_record",
        "source_file", "batch_date", F.current_timestamp().alias("quarantined_at"),
    )
    return good, bad

## Step 3: Load Silver and quarantine

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes the valid orders to Silver and the rejected ones to the quarantine table, then fails the run if too many rows were rejected.<br><br>
<b>Why it matters</b><br>The ratio check is the pipeline's safety valve: a few bad rows are normal, but if half the batch is broken, something is wrong upstream and the run should stop instead of publishing a misleading Gold table.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Silver: 6 valid orders (1001, 1002 with the <b>updated</b> amount 85.00, 1004, 1006, 1007, 1009). Quarantine: 4 rows: 1003 (invalid amount), 1005 (unknown customer), 1008 (invalid timestamp) and the broken line (malformed line).
</div>

In [0]:
def load_silver():
    good, bad = build_silver()
    good.write.mode("overwrite").saveAsTable(CONFIG["silver_orders"])
    bad.write.mode("overwrite").saveAsTable(CONFIG["quarantine_orders"])

    n_good = spark.table(CONFIG["silver_orders"]).count()
    n_bad = spark.table(CONFIG["quarantine_orders"]).count()
    ratio = n_bad / (n_good + n_bad)
    logger.info("silver: %d valid, %d quarantined (%.0f%%)", n_good, n_bad, ratio * 100)
    if ratio > CONFIG["max_bad_ratio"]:
        raise RuntimeError(f"too many bad rows: {ratio:.0%} > {CONFIG['max_bad_ratio']:.0%}")

load_silver()
spark.table(CONFIG["silver_orders"]).orderBy("order_id").show()
spark.table(CONFIG["quarantine_orders"]).select("order_id", "customer_id", "amount", "dq_reasons", "_corrupt_record") \
     .orderBy("order_id").show(truncate=False)

## Step 4: Aggregate into Gold

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Builds two Gold tables from Silver: daily sales per country (excluding cancelled orders, refunds as negative revenue), and the lifetime value of each customer with a rank.<br><br>
<b>Why it matters</b><br>Gold tables answer business questions directly, with clear names and pre-computed metrics, so dashboards don't repeat business logic.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>gold_daily_sales</code>: one row per date and country. For example 2024-11-02 AE = 300.00 and 2024-11-02 UK = -15.00 (a refund). <code>gold_customer_value</code>: Amal ranks first with 330.50 lifetime revenue.
</div>

In [0]:
def load_gold():
    silver = spark.table(CONFIG["silver_orders"]).filter(F.col("status") != "CANCELLED")
    signed_amount = F.when(F.col("status") == "REFUND", -F.col("amount")).otherwise(F.col("amount"))

    daily = (
        silver.groupBy("order_date", "country")
        .agg(F.sum(signed_amount).alias("net_revenue"), F.count("*").alias("orders"),
             F.countDistinct("customer_id").alias("customers"))
        .orderBy("order_date", "country")
    )
    daily.write.mode("overwrite").saveAsTable(CONFIG["gold_daily_sales"])

    value = (
        silver.groupBy("customer_id", "customer_name", "country", "tier")
        .agg(F.sum(signed_amount).alias("lifetime_revenue"), F.count("*").alias("orders"),
             F.max("order_ts").alias("last_order_ts"))
        .withColumn("revenue_rank", F.dense_rank().over(Window.orderBy(F.desc("lifetime_revenue"))))
    )
    value.write.mode("overwrite").saveAsTable(CONFIG["gold_customer_value"])
    logger.info("gold: %d daily rows, %d customers", daily.count(), value.count())

load_gold()
spark.table(CONFIG["gold_daily_sales"]).orderBy("order_date", "country").show()
spark.table(CONFIG["gold_customer_value"]).orderBy("revenue_rank").show()

## Step 5: Reconcile

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Checks that every Bronze row is accounted for: Bronze rows = Silver rows + quarantined rows + duplicates removed. It also checks that Gold revenue matches Silver.<br><br>
<b>Why it matters</b><br>Reconciliation proves the pipeline doesn't lose or invent data. It's the first thing an auditor, or an interviewer, asks about.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Bronze 11 = Silver 6 + quarantine 4 + duplicates 1, so <code>reconciled: True</code>. Gold total equals Silver total, so <code>gold matches silver: True</code>.
</div>

In [0]:
def reconcile():
    bronze = spark.table(CONFIG["bronze_orders"])
    n_bronze = bronze.count()
    # Duplicates = rows with an order_id that appears more than once, minus one kept copy per id
    n_dupes = n_bronze - bronze.select(F.coalesce("order_id", "_corrupt_record")).distinct().count()
    n_silver = spark.table(CONFIG["silver_orders"]).count()
    n_quarantine = spark.table(CONFIG["quarantine_orders"]).count()
    ok_rows = n_bronze == n_silver + n_quarantine + n_dupes
    print(f"bronze {n_bronze} = silver {n_silver} + quarantine {n_quarantine} + duplicates {n_dupes} -> reconciled: {ok_rows}")

    silver_total = (spark.table(CONFIG["silver_orders"]).filter("status != 'CANCELLED'")
                    .select(F.sum(F.when(F.col("status") == "REFUND", -F.col("amount")).otherwise(F.col("amount")))).first()[0])
    gold_total = spark.table(CONFIG["gold_daily_sales"]).agg(F.sum("net_revenue")).first()[0]
    ok_money = silver_total == gold_total
    print(f"silver net revenue {silver_total} | gold net revenue {gold_total} -> gold matches silver: {ok_money}")
    if not (ok_rows and ok_money):
        raise RuntimeError("reconciliation failed")

reconcile()

## Step 6: Run the whole pipeline, twice

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Wraps all steps in one <code>run_pipeline()</code> function with logging and timing, then runs it <b>again</b> with the same input.<br><br>
<b>Why it matters</b><br>This is the idempotency test. If the second run changed any count, the pipeline would double-count after every retry. Because Bronze uses dynamic partition overwrite and Silver/Gold are rebuilt from Bronze, a re-run produces identical results.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Two runs with the same log lines and the same counts (11 Bronze, 6 Silver, 4 quarantined), and <code>identical results after re-run: True</code>.
</div>

In [0]:
import time

def run_pipeline():
    start = time.time()
    logger.info("pipeline start")
    try:
        load_bronze()
        load_silver()
        load_gold()
        reconcile()
    except Exception:
        logger.exception("pipeline failed")
        raise
    logger.info("pipeline finished in %.1fs", time.time() - start)

snapshot = lambda: (spark.table(CONFIG["gold_customer_value"]).orderBy("customer_id").collect(),
                    spark.table(CONFIG["silver_orders"]).count())

run_pipeline()
first = snapshot()
run_pipeline()
print("identical results after re-run:", snapshot() == first)

## Under the hood

```
load_bronze()   read CSV/JSON  ──▶  write (Delta, dynamic partition overwrite)     job(s)
load_silver()   read bronze ─▶ standardize ─▶ window dedup (shuffle by order_id)
                ─▶ join customers (broadcast, small) ─▶ validate ─▶ write silver + quarantine
load_gold()     read silver ─▶ groupBy (shuffle) ─▶ write gold
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> each `write` and `count` runs a job. The pipeline is a sequence of small jobs, one per table.

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Wide</span> deduplication, aggregation and the rank window shuffle. The customer join is broadcast because the table is tiny.

Note that `build_silver()` is used twice (good and bad rows). Without caching, Spark computes the Silver logic for each write, which is why production pipelines often write Silver **once** and derive everything else from the written table. Run the cell to see the Silver plan: look for the `Window` (dedup), the `BroadcastHashJoin` (customers) and the `Exchange` nodes.

In [0]:
good_df, _ = build_silver()
good_df.explain()

In [0]:
# Optional cleanup: drop this lesson's tables and files.
for key in ["bronze_orders", "bronze_customers", "silver_orders", "quarantine_orders", "gold_daily_sales", "gold_customer_value"]:
    spark.sql(f"DROP TABLE IF EXISTS {CONFIG[key]}")
dbutils.fs.rm(BASE, True)

## What you would add in production

| Gap in this lesson | Production approach | Where it's covered |
|---|---|---|
| Re-reads all files every run | Incremental ingestion with **Auto Loader** or `COPY INTO` | `08_streaming_and_ingestion` |
| Silver is fully rebuilt | `MERGE` upserts into Silver | `03_delta_lake/07_merge_upserts` |
| Customer changes overwrite history | SCD Type 2 dimension | `03_delta_lake/07_merge_upserts` |
| Rules coded by hand | Delta constraints and Lakeflow expectations | `03_delta_lake`, `07_workflows` |
| Run manually | A Lakeflow Job with tasks per layer, retries and alerts | `07_workflows` |
| Hard-coded config | Job parameters, widgets, Asset Bundle targets | `04_databricks`, `07_workflows` |
| Functions in a notebook | A Python package with unit tests | `07_workflows/07_testing_and_code_structure` |

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: duplicates in Silver after a re-run</b><br>
Bronze was written with <code>append</code>. Use dynamic partition overwrite on the batch, or <code>MERGE</code> on the key.<br><br>
<b>⛔ Problem: the pipeline fails on one bad value</b><br>A strict <code>cast</code> under ANSI mode. Read Bronze as strings, then convert with <code>try_cast</code> / <code>try_to_timestamp</code> and quarantine.<br><br>
<b>⛔ Problem: rows disappear between Bronze and Silver</b><br>An inner join to a dimension dropped unknown keys. Use a left join and quarantine the unmatched rows, then reconcile counts.<br><br>
<b>⛔ Problem: Gold doesn't match the source system</b><br>Business rules differ (cancelled orders, refunds, time zones). Write the rules down, and reconcile totals at every layer.<br><br>
**⛔ Problem: <code>QUERY_ONLY_CORRUPT_RECORD_COLUMN</code>**<br>Split good and bad rows from the <b>Bronze table</b>, not from the raw file read (lesson 21).<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. Walk me through the medallion architecture.</b><br>
Bronze stores raw data as received, plus audit columns such as source file and load time, so it can be replayed. Silver holds cleaned, typed, deduplicated and validated data, one row per entity, often enriched with dimensions. Gold holds business-level aggregates and marts for reporting. Each layer is built from the previous one.<br><br>

<b>🎤 2. How do you make a batch pipeline idempotent?</b><br>
Make every write replace exactly the data of the batch it processes: dynamic partition overwrite or <code>replaceWhere</code> per batch date, or <code>MERGE</code> on a business key. Then re-running a batch produces the same result instead of duplicates. I test it by running the pipeline twice and comparing outputs.<br><br>

<b>🎤 3. How do you handle bad records without failing the whole pipeline?</b><br>
Land everything in Bronze as strings with a corrupt-record column, convert in Silver with <code>try_*</code> functions, attach reasons to rows that break rules, write them to a quarantine table, and fail the run only if the bad-row ratio exceeds a threshold.<br><br>

<b>🎤 4. How do you deduplicate in Silver?</b><br>
Define the business key and an ordering that identifies the latest version (ingestion date, source file or an update timestamp), then keep <code>row_number() == 1</code> per key.<br><br>

<b>🎤 5. How do you prove the pipeline didn't lose data?</b><br>
Reconciliation: Bronze rows must equal Silver rows plus quarantined rows plus removed duplicates, and Gold totals must match Silver totals. The run fails if they don't.<br><br>

<b>🎤 6. What would you change to run this on large, daily-growing data?</b><br>
Ingest incrementally with Auto Loader, upsert Silver with <code>MERGE</code>, partition or cluster tables by date, orchestrate the layers as tasks in a Lakeflow Job with retries and alerts, and move the functions into a tested Python package.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. In the medallion architecture, which layer typically holds deduplicated, validated, and enriched records at the level of individual business entities?</b><br>
A. Bronze<br>
B. Silver<br>
C. Gold<br>
D. Landing<br>
<details><summary><b>Show answer</b></summary><b>B.</b> Bronze is raw, and Gold is aggregated for business use.</details><br><br>

<b>Q2. A daily job re-processes a date after a failure and the target table ends up with duplicate rows for that date. Which change fixes this while keeping other dates?</b><br>
A. Use <code>mode("append")</code> with <code>ignoreDuplicates</code><br>
B. Use <code>MERGE INTO</code> on the business key, or overwrite only that date's data<br>
C. Run <code>OPTIMIZE</code> after each load<br>
D. Use a temp view instead of a table<br>
<details><summary><b>Show answer</b></summary><b>B.</b> Both make the load idempotent. <code>OPTIMIZE</code> compacts files and doesn't remove duplicates.</details><br><br>

<b>Q3. Where should raw data be preserved exactly as received, so that downstream tables can be rebuilt?</b><br>
A. Gold<br>
B. Silver<br>
C. Bronze<br>
D. A materialized view<br>
<details><summary><b>Show answer</b></summary><b>C.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Add a third orders file <code>orders_2024-11-03.csv</code> with 3 new orders, one of them a refund for customer C4 and one with currency <code>usd</code> in lower case. Run <code>run_pipeline()</code> and check the new Gold rows</li><li>Add a validation rule: <code>currency</code> must be one of <code>INR</code>, <code>GBP</code>, <code>AED</code>, <code>USD</code></li><li>Add a Gold table <code>gold_tier_summary</code> with net revenue and order count per customer tier</li><li>Make the reconciliation also check that no <code>order_id</code> appears twice in Silver</li><li>In two sentences, explain to an interviewer why Bronze stores every column as a string</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. A new day of data
write_file(f"{CONFIG['landing_orders']}/orders_2024-11-03.csv", """order_id,customer_id,order_ts,amount,currency,status
1010,C4,2024-11-03 09:00:00,50.00,AED,REFUND
1011,C2,2024-11-03 10:30:00,99.00,usd,PAID
1012,C3,2024-11-03 11:45:00,120.00,AED,PAID
""")

# 2. Extend validate() with a currency rule (wraps the original function)
CURRENCIES = ["INR", "GBP", "AED", "USD"]
_base_validate = validate
def validate(df):
    checked = _base_validate(df)
    return checked.withColumn("dq_reasons", F.array_compact(F.concat(
        "dq_reasons", F.array(F.when(~F.col("currency").isin(CURRENCIES), F.lit("unknown currency"))))))

run_pipeline()
spark.table(CONFIG["gold_daily_sales"]).filter("order_date = '2024-11-03'").show()

# 3. Tier summary from Silver
silver = spark.table(CONFIG["silver_orders"]).filter("status != 'CANCELLED'")
signed = F.when(F.col("status") == "REFUND", -F.col("amount")).otherwise(F.col("amount"))
tier_summary = silver.groupBy("tier").agg(F.sum(signed).alias("net_revenue"), F.count("*").alias("orders"))
tier_summary.write.mode("overwrite").saveAsTable(f"{DB}.lesson30_gold_tier_summary")
spark.table(f"{DB}.lesson30_gold_tier_summary").orderBy("tier").show()

# 4. Uniqueness check in Silver
s = spark.table(CONFIG["silver_orders"])
assert s.count() == s.select("order_id").distinct().count(), "duplicate order_id in silver"
print("silver keys unique: True")

# 5. Strings never fail to parse, so every record that arrives is preserved in Bronze, even broken ones.
#    Typing happens in Silver, where bad values can be detected, quarantined and replayed from Bronze.

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Medallion: <b>Bronze</b> (raw + audit columns) → <b>Silver</b> (typed, deduplicated, validated, enriched) → <b>Gold</b> (business aggregates)</li><li>Read Bronze with explicit all-string schemas and a corrupt-record column. Type in Silver with <code>try_*</code></li><li>Deduplicate with <code>row_number()</code> over the business key, latest version first</li><li>Validate with a reasons array, quarantine bad rows, fail above a bad-row threshold</li><li>Make writes idempotent (dynamic partition overwrite, <code>replaceWhere</code>, <code>MERGE</code>) and prove it with a re-run</li><li>Reconcile counts and totals between layers</li><li>Small named functions, config in one place, logging at each step</li></ul>
</div>

| Step | Code |
|---|---|
| Bronze read | `spark.read.schema(all_strings).option("columnNameOfCorruptRecord", ...)` |
| Lineage | `F.col("_metadata.file_name")`, `F.current_timestamp()` |
| Idempotent write | `.option("partitionOverwriteMode", "dynamic").partitionBy("batch_date")` |
| Safe typing | `try_cast`, `try_to_timestamp` |
| Dedup | `row_number().over(Window.partitionBy(key).orderBy(desc(version)))` |
| Quarantine | `F.array_compact(F.array(*[F.when(rule, F.lit(name))]))` |
| Reconcile | `bronze == silver + quarantine + duplicates` |

## Git commit

```
git add 01_spark_basics/30_building_an_etl_pipeline.ipynb
git commit -m "add 30 building an etl pipeline notebook"
```